In [67]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split,RandomizedSearchCV
from sklearn.preprocessing import StandardScaler,LabelEncoder
from sklearn.ensemble import RandomForestClassifier,GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import confusion_matrix,classification_report,ConfusionMatrixDisplay
from xgboost import XGBClassifier

In [47]:
from google.colab import files

uploaded = files.upload()

Saving bank.zip to bank (1).zip


In [57]:
import zipfile
import os

with zipfile.ZipFile("bank.zip", "r") as zip_ref:
    zip_ref.extractall("bank_data")

print(os.listdir("bank_data"))

['bank-full.csv', 'bank.csv', 'bank-names.txt']


In [58]:
df = pd.read_csv("bank_data/bank-full.csv",sep=";")
df.head()

,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome,y
0,58,management,married,tertiary,no,2143,yes,no,unknown,5,may,261,1,-1,0,unknown,no
1,44,technician,single,secondary,no,29,yes,no,unknown,5,may,151,1,-1,0,unknown,no
2,33,entrepreneur,married,secondary,no,2,yes,yes,unknown,5,may,76,1,-1,0,unknown,no
3,47,blue-collar,married,unknown,no,1506,yes,no,unknown,5,may,92,1,-1,0,unknown,no
4,33,unknown,single,unknown,no,1,no,no,unknown,5,may,198,1,-1,0,unknown,no


In [50]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 45211 entries, 0 to 45210
Data columns (total 17 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   age        45211 non-null  int64 
 1   job        45211 non-null  object
 2   marital    45211 non-null  object
 3   education  45211 non-null  object
 4   default    45211 non-null  object
 5   balance    45211 non-null  int64 
 6   housing    45211 non-null  object
 7   loan       45211 non-null  object
 8   contact    45211 non-null  object
 9   day        45211 non-null  int64 
 10  month      45211 non-null  object
 11  duration   45211 non-null  int64 
 12  campaign   45211 non-null  int64 
 13  pdays      45211 non-null  int64 
 14  previous   45211 non-null  int64 
 15  poutcome   45211 non-null  object
 16  y          45211 non-null  object
dtypes: int64(7), object(10)
memory usage: 5.9+ MB


In [59]:
print(df['marital'].unique())
print(df['education'].unique())
print(df['default'].unique())
print(df['housing'].unique())
print(df['loan'].unique())
print(df['y'].unique())

['married' 'single' 'divorced']
['tertiary' 'secondary' 'unknown' 'primary']
['no' 'yes']
['yes' 'no']
['no' 'yes']
['no' 'yes']


In [61]:
l = LabelEncoder()
df['marital'] = l.fit_transform(df['marital'])
df['education'] = l.fit_transform(df['education'])
df['default'] = l.fit_transform(df['default'])
df['housing'] = l.fit_transform(df['housing'])
df['loan'] = l.fit_transform(df['loan'])
df['y'] = l.fit_transform(df['y'])
df.head()

,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome,y
0,58,management,1,2,0,2143,1,0,unknown,5,may,261,1,-1,0,unknown,0
1,44,technician,2,1,0,29,1,0,unknown,5,may,151,1,-1,0,unknown,0
2,33,entrepreneur,1,1,0,2,1,1,unknown,5,may,76,1,-1,0,unknown,0
3,47,blue-collar,1,3,0,1506,1,0,unknown,5,may,92,1,-1,0,unknown,0
4,33,unknown,2,3,0,1,0,0,unknown,5,may,198,1,-1,0,unknown,0


In [62]:
df = df.drop(['contact','month','poutcome'],axis = 1)
df.head()

,age,job,marital,education,default,balance,housing,loan,day,duration,campaign,pdays,previous,y
0,58,management,1,2,0,2143,1,0,5,261,1,-1,0,0
1,44,technician,2,1,0,29,1,0,5,151,1,-1,0,0
2,33,entrepreneur,1,1,0,2,1,1,5,76,1,-1,0,0
3,47,blue-collar,1,3,0,1506,1,0,5,92,1,-1,0,0
4,33,unknown,2,3,0,1,0,0,5,198,1,-1,0,0


In [63]:
df = pd.get_dummies(df,columns=['job'],dtype='int64')
df.head()

,age,marital,education,default,balance,housing,loan,day,duration,campaign,pdays,previous,y,job_admin.,job_blue-collar,job_entrepreneur,job_housemaid,job_management,job_retired,job_self-employed,job_services,job_student,job_technician,job_unemployed,job_unknown
0,58,1,2,0,2143,1,0,5,261,1,-1,0,0,0,0,0,0,1,0,0,0,0,0,0,0
1,44,2,1,0,29,1,0,5,151,1,-1,0,0,0,0,0,0,0,0,0,0,0,1,0,0
2,33,1,1,0,2,1,1,5,76,1,-1,0,0,0,0,1,0,0,0,0,0,0,0,0,0
3,47,1,3,0,1506,1,0,5,92,1,-1,0,0,0,1,0,0,0,0,0,0,0,0,0,0
4,33,2,3,0,1,0,0,5,198,1,-1,0,0,0,0,0,0,0,0,0,0,0,0,0,1


In [64]:
ss = StandardScaler()
df['age'] = ss.fit_transform(df[['age']])
df['balance'] = ss.fit_transform(df[['balance']])
df['day'] = ss.fit_transform(df[['day']])
df['duration'] = ss.fit_transform(df[['duration']])
df['campaign'] = ss.fit_transform(df[['campaign']])
df['pdays'] = ss.fit_transform(df[['pdays']])
df.head()

,age,marital,education,default,balance,housing,loan,day,duration,campaign,pdays,previous,y,job_admin.,job_blue-collar,job_entrepreneur,job_housemaid,job_management,job_retired,job_self-employed,job_services,job_student,job_technician,job_unemployed,job_unknown
0,1.606965,1,2,0,0.256419,1,0,-1.298476,0.011016,-0.569351,-0.411453,0,0,0,0,0,0,1,0,0,0,0,0,0,0
1,0.288529,2,1,0,-0.437895,1,0,-1.298476,-0.416127,-0.569351,-0.411453,0,0,0,0,0,0,0,0,0,0,0,1,0,0
2,-0.747384,1,1,0,-0.446762,1,1,-1.298476,-0.707361,-0.569351,-0.411453,0,0,0,0,1,0,0,0,0,0,0,0,0,0
3,0.571051,1,3,0,0.047205,1,0,-1.298476,-0.645231,-0.569351,-0.411453,0,0,0,1,0,0,0,0,0,0,0,0,0,0
4,-0.747384,2,3,0,-0.447091,0,0,-1.298476,-0.233620,-0.569351,-0.411453,0,0,0,0,0,0,0,0,0,0,0,0,0,1


In [65]:
x = df.drop('y',axis=1)
y = df['y']

In [66]:
x_train,x_test,y_train,y_test = train_test_split(x,y,test_size=0.2,random_state=42,stratify=y)

In [103]:
lg = Pipeline([
    ('lg',LogisticRegression(random_state=42,max_iter=1000))
])

rf = Pipeline([
    ('rf',RandomForestClassifier(n_estimators=400,random_state=42,n_jobs = -1,class_weight = 'balanced'))
])


gb = Pipeline([
    ('gb',GradientBoostingClassifier(random_state=42,max_depth=3))
])

xgb = Pipeline([
    ('xgb',XGBClassifier(n_estimators=300,
        learning_rate=0.05,
        max_depth=4,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42,
        eval_metric="logloss"))
])

In [112]:
lg.fit(x_train,y_train)

Pipeline(steps=[('lg', LogisticRegression(max_iter=1000, random_state=42))])

In [74]:
y_pred = lg.predict(x_test)

In [75]:
lg.score(x_train,y_train)

0.8907597876575979

In [77]:
lg.score(x_test,y_test)

0.8900807254229791

In [111]:
print(classification_report(y_test,y_pred))

              precision    recall  f1-score   support

           0       0.90      0.98      0.94      7985
           1       0.59      0.21      0.30      1058

    accuracy                           0.89      9043
   macro avg       0.74      0.59      0.62      9043
weighted avg       0.87      0.89      0.87      9043



In [110]:
rf.fit(x_train,y_train)

Pipeline(steps=[('rf',
                 RandomForestClassifier(class_weight='balanced',
                                        n_estimators=400, n_jobs=-1,
                                        random_state=42))])

In [94]:
rf_pred = rf.predict(x_test)

In [95]:
rf.score(x_train,y_train)

1.0

In [96]:
rf.score(x_test,y_test)

0.8945040362711489

In [97]:
print(classification_report(y_test,rf_pred))

              precision    recall  f1-score   support

           0       0.91      0.98      0.94      7985
           1       0.62      0.25      0.36      1058

    accuracy                           0.89      9043
   macro avg       0.76      0.62      0.65      9043
weighted avg       0.87      0.89      0.87      9043



In [109]:
gb.fit(x_train,y_train)

Pipeline(steps=[('gb', GradientBoostingClassifier(random_state=42))])

In [100]:
gb.score(x_train,y_train)

0.9041694315416943

In [101]:
gb.score(x_test,y_test)

0.8948357845847617

In [102]:
print(classification_report(y_test,gb.predict(x_test)))

              precision    recall  f1-score   support

           0       0.92      0.97      0.94      7985
           1       0.59      0.32      0.42      1058

    accuracy                           0.89      9043
   macro avg       0.75      0.65      0.68      9043
weighted avg       0.88      0.89      0.88      9043



In [104]:
xgb.fit(x_train,y_train)

Pipeline(steps=[('xgb',
                 XGBClassifier(base_score=None, booster=None, callbacks=None,
                               colsample_bylevel=None, colsample_bynode=None,
                               colsample_bytree=0.8, device=None,
                               early_stopping_rounds=None,
                               enable_categorical=True, eval_metric='logloss',
                               feature_types=None, feature_weights=None,
                               gamma=None, grow_policy=None,
                               importance_type=None,
                               interaction_constraints=None, learning_rate=0.05,
                               max_bin=None, max_cat_threshold=None,
                               max_cat_to_onehot=None, max_delta_step=None,
                               max_depth=4, max_leaves=None,
                               min_child_weight=None, missing=nan,
                               monotone_constraints=None, multi_strategy=None,
                               n_estimators=300, n_jobs=None,
                               num_parallel_tree=None, ...))])

In [105]:
xgb.score(x_train,y_train)

0.9109433753594337

In [106]:
xgb.score(x_test,y_test)

0.8972686055512551

In [107]:
print(classification_report(y_test,xgb.predict(x_test)))

              precision    recall  f1-score   support

           0       0.92      0.97      0.94      7985
           1       0.60      0.36      0.45      1058

    accuracy                           0.90      9043
   macro avg       0.76      0.66      0.70      9043
weighted avg       0.88      0.90      0.89      9043



In [113]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

models = {
    "Logistic Regression": lg,
    "Random Forest": rf,
    "Gradient Boosting": gb,
    "XGBoost": xgb
}

for name, model in models.items():

    pred = model.predict(x_test)
    prob = model.predict_proba(x_test)[:, 1]

    print(name)
    print("Accuracy :", accuracy_score(y_test, pred))
    print("Precision:", precision_score(y_test, pred))
    print("Recall   :", recall_score(y_test, pred))
    print("F1       :", f1_score(y_test, pred))
    print("ROC-AUC  :", roc_auc_score(y_test, prob))
    print("-" * 40)

Logistic Regression
Accuracy : 0.8900807254229791
Precision: 0.5860215053763441
Recall   : 0.2060491493383743
F1       : 0.3048951048951049
ROC-AUC  : 0.8649521254999628
----------------------------------------
Random Forest
Accuracy : 0.8945040362711489
Precision: 0.619815668202765
Recall   : 0.2542533081285444
F1       : 0.3605898123324397
ROC-AUC  : 0.894738835695
----------------------------------------
Gradient Boosting
Accuracy : 0.8948357845847617
Precision: 0.5930434782608696
Recall   : 0.3223062381852552
F1       : 0.417636252296387
ROC-AUC  : 0.8943752641117028
----------------------------------------
XGBoost
Accuracy : 0.8972686055512551
Precision: 0.6022187004754358
Recall   : 0.3591682419659735
F1       : 0.4499703966844287
ROC-AUC  : 0.8994240737299259
----------------------------------------
